# RAG Query Engine and Persistent Chatbot

This notebook is now organized like a small RAG application instead of a pile of helper functions. The same behavior is kept, but the responsibilities are grouped into three modules:

1. `ChromaKnowledgeBase`: reconnects to ChromaDB, rebuilds the LlamaIndex vector index, rebuilds BM25, and creates the hybrid retriever.
2. `JsonChatSessionStore`: saves and loads chat memory as one JSON file per chat in the `session` folder.
3. `RagNewsChatbot`: exposes the product-facing actions: ask one question, open a chat, continue a chat, inspect history, and show sources.

```mermaid
flowchart LR
    U[User question] --> APP[RagNewsChatbot]
    APP --> HR[HybridRetriever]
    HR --> VS[Semantic search\nChroma vector index]
    HR --> BM[Keyword search\nBM25]
    VS --> RRF[Reciprocal rank fusion]
    BM --> RRF
    RRF --> LLM[Gemma 3 1B via Ollama]
    LLM --> A[Answer + sources]
    APP <--> MEM[ChatMemoryBuffer]
    MEM <--> JSON[session/*.json]
```

The important product idea is separation of concerns: retrieval, generation, and persistence are related, but they should not be tangled together.

In [ ]:
# Run this once if the LlamaIndex integrations are missing, then restart the kernel.
%pip install llama-index-vector-stores-chroma llama-index-retrievers-bm25 llama-index-llms-ollama

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if not (project_root / "app.py").exists() and (project_root.parent / "app.py").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import importlib
import src.rag.chatbot as rag_chatbot_module
import src.rag.factory as rag_factory_module
import src.rag.session_store as rag_session_store_module

importlib.reload(rag_session_store_module)
importlib.reload(rag_chatbot_module)
importlib.reload(rag_factory_module)

from src.rag import (
    DEFAULT_COLLECTION_NAME,
    DEFAULT_EMBED_MODEL_NAME,
    DEFAULT_HUGGINGFACE_MODEL_KEY,
    DEFAULT_LLM_PROVIDER,
    DEFAULT_OLLAMA_MODEL,
    HUGGINGFACE_CHAT_MODELS,
    LLM_PROVIDER_HUGGINGFACE,
    LLM_PROVIDER_OLLAMA,
    ChromaKnowledgeBase,
    JsonChatSessionStore,
    create_rag_app,
    default_paths,
    print_sources,
    print_response
)

resource module not available on Windows


In [2]:
PROJECT_ROOT = project_root
paths = default_paths(PROJECT_ROOT)
NEWS_DIR = paths.news_dir
CHROMA_DIR = paths.chroma_dir
SESSION_DIR = paths.session_dir
COLLECTION_NAME = DEFAULT_COLLECTION_NAME
EMBED_MODEL_NAME = DEFAULT_EMBED_MODEL_NAME
LLM_PROVIDER = DEFAULT_LLM_PROVIDER  # Choose: LLM_PROVIDER_OLLAMA or LLM_PROVIDER_HUGGINGFACE
OLLAMA_MODEL = DEFAULT_OLLAMA_MODEL
HUGGINGFACE_MODEL = DEFAULT_HUGGINGFACE_MODEL_KEY  # deepseek_v3, minimax_m3, qwen_3_5, or any full HF model id
HUGGINGFACE_PROVIDER = "auto"

print(f"Project root: {PROJECT_ROOT}")
print(f"News source folder: {NEWS_DIR}")
print(f"ChromaDB folder: {CHROMA_DIR}")
print(f"Session folder: {SESSION_DIR}")
print(f"Collection name: {COLLECTION_NAME}")
print(f"Embedding model: {EMBED_MODEL_NAME}")
print(f"LLM provider: {LLM_PROVIDER}")
print(f"Ollama model: {OLLAMA_MODEL}")
print(f"Hugging Face model: {HUGGINGFACE_CHAT_MODELS.get(HUGGINGFACE_MODEL, HUGGINGFACE_MODEL)}")

Project root: C:\Program Files\Studying\coding\RAG_project
News source folder: C:\Program Files\Studying\coding\RAG_project\data\hk_free_press_news
ChromaDB folder: C:\Program Files\Studying\coding\RAG_project\chromadb_store
Session folder: C:\Program Files\Studying\coding\RAG_project\session
Collection name: news_chat
Embedding model: BAAI/bge-small-en-v1.5
LLM provider: huggingface
Ollama model: gemma3:1b
Hugging Face model: deepseek-ai/DeepSeek-V3-0324


In [ ]:
# Retrieval, fusion, and knowledge-base classes are now centralized in src/rag/.
# This notebook imports and uses the same implementation as app.py.

In [ ]:
# Session persistence and chatbot facade now come from src/rag/.
# This removes duplicate maintenance between the notebook and app.py.

## Build the Application Objects

Run the next cell once per notebook session. It creates:

- `knowledge_base`: reconnects to the persisted `news_chat` ChromaDB collection
- `session_store`: reads and writes chat JSON files under `session/`
- `rag_app`: the single object you use for query, chat, sources, and history

This is the main readability improvement: examples below call methods on `rag_app` instead of manually coordinating many functions and global variables.

## Run the RAG App

The next cells demonstrate the same functionality as before, but through one object: `rag_app`.

Use `rag_app.ask(...)` for one-off questions. Use `rag_app.open_chat(...)` and `rag_app.chat(...)` for multi-turn conversations with persistent memory.

### Initialize the App

In [3]:
# Run this once after reopening the notebook.
# It reconnects to ChromaDB, rebuilds semantic + BM25 retrieval, and prepares chat persistence.
rag_app = create_rag_app(
    chroma_dir=CHROMA_DIR,
    session_dir=SESSION_DIR,
    news_dir=NEWS_DIR,
    collection_name=COLLECTION_NAME,
    embed_model_name=EMBED_MODEL_NAME,
    llm_provider=LLM_PROVIDER,
    ollama_model=OLLAMA_MODEL,
    huggingface_model=HUGGINGFACE_MODEL,
    huggingface_provider=HUGGINGFACE_PROVIDER,
    final_top_k=5,
)
knowledge_base = rag_app.knowledge_base

print(f"Stored vector count: {knowledge_base.count()}")
print("RAG app is ready.")

Stored vector count: 1499
RAG app is ready.


### Single-turn RAG: use this when the question does not need chat history.

In [ ]:
sample_query = "What model you are ? and what is the latest information you can provide to me ?"
query_response = rag_app.ask(sample_query)

print("-" * 70)
print_sources(query_response)

### Multi-turn RAG With Saved Chat History

`rag_app.open_chat(...)` loads the saved JSON file when it exists, or creates a new one when it does not. Every `rag_app.chat(...)` call saves the updated conversation automatically.

In [ ]:
# Create or load one persistent chat session.
# If session/Sanae_Takaichi.json exists, this restores its previous messages.
chat_id = rag_app.open_chat("Donald Trump")

chat_response = rag_app.chat(
    chat_id=chat_id,
    message="What model you are ? and what is the latest information you can provide to me ?",
)

print_response(chat_response)
print('-' * 70)
print_sources(chat_response)

In [ ]:
# Follow-up question: this uses the same saved chat memory, so "she" refers to Sanae Takaichi.
follow_up_response = rag_app.chat(
    chat_id=chat_id,
    message="How long did she talk with Donald Trump during a phone call?",
)
print_sources(follow_up_response)

In [ ]:
# Inspect the current in-memory history for this chat.
rag_app.show_history(chat_id)

In [ ]:
# List saved chat files on disk.
rag_app.list_saved_chats()

### Reopen an Existing Chat

After reopening the notebook, run the import/config/module cells and the app initialization cell first. Then `rag_app.open_chat("Sanae Takaichi")` restores the saved conversation from `session/Sanae_Takaichi.json`.

In [4]:
loaded_chat_id = rag_app.open_chat("Donald Trump")

In [5]:
rag_app.show_history(loaded_chat_id)

user: What model you are ? and what is the latest information you can provide to me ?
assistant: I am an AI assistant designed to help you retrieve and analyze information from the Hong Kong Free Press (HKFP) news knowledge base. My responses are based on the documents I have access to, and I cannot provide information outside of that context.

### Latest Information I Can Provide (Based on Retrieved Documents):

1. **Lamppost Adornments in Sha Tin (Dec 2026)**  
   - The article discusses new patriotic decorations on lampposts celebrating China's 15th five-year plan, though the messaging is criticized for being unclear to the public.  
   - Historical skepticism about five-year plans is mentioned, referencing past famines and economic planning failures.  

2. **Corporate Carbon Reporting (Jan 2026)**  
   - Hong Kong-listed companies must now mandatorily disclose **Scope 1 and 2 emissions** (direct and electricity-related emissions) starting in 2026, with **Scope 3** (supply chain emi

In [ ]:
# Continue a loaded chat without rerunning the original conversation cells.
reloaded_response = rag_app.chat(
    chat_id=loaded_chat_id,
    message="Did he is a most powerful man in the world ?. In the above sentence, can you guess who is he ?",
)

print_response(reloaded_response)
print('-' * 70)
print_sources(reloaded_response)

In [10]:
print_response(reloaded_response)
print('-' * 70)
print_sources(reloaded_response)

In the sentence you provided ("Did he is a most powerful man in the world?"), the pronoun **"he"** likely refers to **Donald Trump**, based on the context of our conversation and the retrieved HKFP documents.  

### Why Trump?  
1. **Recent Relevance**: Our discussion has focused on Trump’s actions in 2026 (e.g., trade policies, comments on Jimmy Lai, tensions with China).  
2. **Global Influence**: The documents highlight his role in U.S.-China relations, where he engages directly with Xi Jinping and impacts Hong Kong through trade and geopolitical decisions.  

However, the title of "most powerful man in the world" is subjective and depends on metrics like political, economic, or military influence. The HKFP articles do not explicitly rank global leaders’ power but frame Trump as a consequential figure in shaping Hong Kong’s external pressures.  

Let me know if you’d like analysis of other leaders mentioned (e.g., Xi Jinping, Putin)!
-------------------------------------------------

### Session Management Helpers

Use these APIs to inspect and manage persisted chat sessions:
- `rag_app.count_chat_ids()`
- `rag_app.list_chat_ids()`
- `rag_app.rename_chat(old_chat_id, new_chat_id)`
- `rag_app.delete_chat(chat_id)`

In [11]:
# Count and list chat ids currently saved on disk.
print("Saved chat count:", rag_app.count_chat_ids())
print("Saved chat ids:", rag_app.list_chat_ids())

Saved chat count: 2
Saved chat ids: ['Donald Trump', 'Sanae_Takaichi']


In [ ]:
for delete_chat_id in ['trial sesson']:
    rag_app.delete_chat(delete_chat_id, close_open_session=True, missing_ok=False)